# 01-lossless-structure — 아무것도 안 버리고 줄이기

압축이라고 하면 보통 "덜 중요한 걸 버린다" 를 떠올립니다. 이 랩은 반대입니다.
**아무것도 안 버리고** 토큰만 줄입니다.

가능한 이유는 텍스트에 **내용이 아닌데도 토큰을 먹는 부분**이 있기 때문입니다.

| 무엇이 중복인가 | 어디에 |
|---|---|
| 레코드마다 반복되는 **키 이름** | JSON 배열, API 응답 |
| 줄마다 반복되는 **타임스탬프·서비스명** | 로그 |
| 사람이 보라고 넣은 **들여쓰기** | JSON, XML |
| 세로줄을 맞추려고 채운 **정렬 공백** | 마크다운 표, 설정 파일 |

산문에는 그런 중복이 없습니다. 그래서 같은 코드가 구조화 텍스트에서는
28% 줄이고 산문에서는 1% 줄입니다. **입력이 결과를 정합니다.**

## 1. kit 과 변환 모듈 불러오기

In [ ]:
import sys
from pathlib import Path

LAB = Path.cwd().resolve()
LABS = LAB.parents[0]                  # labs/<이 랩> -> labs
sys.path.insert(0, str(LABS))
sys.path.insert(0, str(LAB))           # 이 랩의 모듈(transforms, blocks 등)

from kit import VERSION, config as C, dataset, env, metrics, tokens as T
from kit.display import table, pct
from kit.runner import Run

# .env 는 labs/.env → 저장소 루트 .env → scripts/explore/.env 순으로 찾습니다.
env.load(verbose=True)

RUNS = LABS.parent / "runs"
print("kit", VERSION, "· 랩", LAB.name)

import transforms as X
from compress import compress, verify_steps

table(
    ["변환", "하는 일", "검증 방법"],
    [[n, t.note,
      "되돌리기" if t.restore else ("정규형 비교" if t.canon else "없음 (손실)")]
     for n, t in X.REGISTRY.items()],
    align=["left", "left", "left"],
    title="등록된 변환",
    note="검증 방법이 없는 변환은 손실로 분류합니다. 무손실은 증명할 수 있어야 합니다.",
)

## 2. 무손실을 말이 아니라 코드로 정의하기

"줄었다" 는 쉽게 보입니다. 어려운 건 **"아무것도 안 잃었다" 를 증명**하는
것입니다. 그래서 모든 변환은 자기를 검증하는 방법을 함께 들고 옵니다.

| 방법 | 어떻게 | 쓰는 변환 |
|---|---|---|
| `restore` | 되돌려서 원본과 **글자 단위**로 비교 | `log_dedup` |
| `canon` | 양쪽을 정규형으로 바꿔 비교 | `json_*`, `xml_*`, `md_table_*`, `kv_*` |
| (없음) | 검증 불가 → **손실로 분류** | `ws_collapse` |

정규형 비교가 필요한 이유는, 들여쓰기를 지우면 **되돌릴 수는 없지만 잃은
정보는 없기** 때문입니다. JSON 의 공백은 내용이 아니므로 파싱한 객체가
같으면 같습니다.

In [ ]:
import json

before = json.dumps([{"id": "A-1", "amount": 1200, "status": "paid"},
                     {"id": "A-2", "amount": 3400, "status": "refunded"}],
                    ensure_ascii=False, indent=2)

after, meta = compress(before, pipeline=["json_to_table"])
ok, why, checked = verify_steps(meta["_steps"])

print("── 압축 전 ──"); print(before)
print("\n── 압축 후 ── (\\x1f 는 눈에 안 보이는 구분자입니다)")
print(after.replace("\x1f", " | "))
print(f"\n검증: {ok} · {why}")
print(f"키 이름이 레코드마다 반복되던 것이 헤더 한 줄로 갔습니다.")

## 3. 검사가 진짜 잡는지 확인하기

**일부러 고장 낸 입력으로 확인하지 않은 검사는 믿지 마세요.**
"동작하는 것처럼 보이지만 아무것도 안 하는" 검사는 없느니만 못합니다.

값을 몰래 지우는 변환을 심어서, 검증이 이걸 잡아내는지 봅니다.

In [ ]:
def sneaky(text):
    """status 필드를 몰래 버리는 변환. 무손실인 척합니다."""
    o = json.loads(text)
    for r in o:
        r.pop("status", None)
    return True, json.dumps(o, ensure_ascii=False, separators=(",", ":")), {}


X.REGISTRY["sneaky"] = X.Transform("sneaky", sneaky, canon=X.json_canon)

_, bad = compress(before, pipeline=["sneaky"])
print("정상 변환:", verify_steps(compress(before, pipeline=["json_to_table"])[1]["_steps"]))
print("몰래 삭제:", verify_steps(bad["_steps"]))

del X.REGISTRY["sneaky"]
print("\n검사가 잡았습니다. 잡지 못했다면 이 랩의 '무손실' 은 빈말이 됩니다.")

## 4. 이 랩의 모든 조건 돌려보기

**조건 1개 = 파일 1개**입니다. `configs/` 를 훑으면 이 랩이 답할 수 있는
질문이 전부 나옵니다. 설정을 새로 추가해도 이 셀은 고칠 필요가 없습니다.

각 조건은 `runs/01-lossless-structure/<설정이름>/<시각>/` 에 따로 기록됩니다. 나중에
"그때 무엇을 돌렸나" 를 설정 이름만 보고 알 수 있게 하려는 것입니다.

| 설정 | 입력 | 무엇을 보려고 |
|---|---|---|
| `structure` | 구조화 12건 | 이 랩의 기본 조건 |
| `prose` | 산문 12건 | **대조군** — 같은 코드가 산문에서 무엇을 하나 |
| `structure-lossy-ws` | 구조화 12건 | 공백까지 접으면 얼마를 더 얻고 무엇을 잃나 |

In [ ]:
def run_config(path):
    cfg = C.load(path)
    cases = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
    counter = T.make_counter(cfg.tokenizer, cfg.model)

    run = Run(cfg, RUNS)
    broken, applied_count, untouched = [], {}, 0
    n_checked = n_unchecked = 0

    for c in cases:
        after, extra = compress(c.text, **cfg.params)
        steps = extra.pop("_steps")
        ok, why, checked = verify_steps(steps)
        extra["verified"] = why
        n_checked += checked
        n_unchecked += len(steps) - checked
        if not ok:
            broken.append((c.id, why))
        for n in extra["applied"]:
            applied_count[n] = applied_count.get(n, 0) + 1
        if not extra["applied"]:
            untouched += 1
        run.add(metrics.per_case(c.id, c.kind, c.text, after, c.must_include,
                                 counter, extra),
                before=c.text, after=after)

    m = metrics.aggregate(run.records, counter)
    m.update({"dataset_name": Path(cfg.dataset["path"]).name,
              "applied_count": applied_count, "untouched": untouched,
              "steps_verified": n_checked, "steps_unverified": n_unchecked,
              "broken": broken})
    return cfg, m, run.finish(m, [f"적용 횟수 {applied_count or '없음'}"])


results = []
for p in sorted(Path("configs").glob("*.yaml")):
    cfg, m, out = run_config(p)
    results.append((cfg.name, m, out))
    flag = " ← 검증 불가 포함" if m["steps_unverified"] else ""
    print(f'{cfg.name:22s} 절감 {m["saved"]:6.1%} · '
          f'검증 {m["steps_verified"]:2d}단계 · 손 안 댐 {m["untouched"]:2d}건{flag}')

## 5. 조건 비교

같은 코드에 조건만 바꿔 돌린 결과입니다. **숫자 하나가 아니라 표를 보세요.**
어떤 조건에서 무엇을 얻고 무엇을 잃는지가 이 랩의 결론입니다.

**여기서 읽어야 할 것 두 가지입니다.**

1. `structure` 와 `prose` 는 **코드가 한 글자도 안 다릅니다.** 입력만 다릅니다.
2. `structure-lossy-ws` 는 절감이 조금 늘지만 **검증 불가 단계**가 생깁니다.
   그만큼 "무손실" 이라는 말을 쓸 수 없게 됩니다.

In [ ]:
table(
    ["설정", "코퍼스", "절감", "최저 보존율", "검증", "검증 불가", "손 안 댐"],
    [[n, m["dataset_name"], pct(m["saved"]), pct(m.get("survival_worst")),
      f'{m["steps_verified"]}단계',
      f'{m["steps_unverified"]}단계' if m["steps_unverified"] else "없음",
      f'{m["untouched"]}건']
     for n, m, _ in results],
    align=["left", "left", "right", "right", "right", "right", "right"],
    title="조건 비교",
    note="검증 불가 단계가 하나라도 있으면 그 조건의 결과는 무손실이 아닙니다.",
)

for n, m, _ in results:
    if m["broken"]:
        print(f"✗ {n}: 정보 손실 {len(m['broken'])}건 — {m['broken'][:2]}")

by = {n: m for n, m, _ in results}
if "structure" in by and "prose" in by:
    print(f'같은 코드, 다른 입력: 구조화 {by["structure"]["saved"]:.1%} '
          f'vs 산문 {by["prose"]["saved"]:.1%}')
if "structure" in by and "structure-lossy-ws" in by:
    d = by["structure-lossy-ws"]["saved"] - by["structure"]["saved"]
    print(f'공백까지 접어서 더 얻은 것: {d:.1%}p — '
          f'그 대가로 무손실 보장을 잃습니다.')

## 6. 유형별 — 어디서 이득이 나나

무손실의 이득은 **중복의 양에 비례**합니다. 그래서 유형마다 크게 다릅니다.

In [ ]:
m = by.get("structure") or results[0][1]

table(
    ["유형", "건수", "절감", "왜"],
    [[k, v["n"], pct(v["saved"]), {
        "json-array": "레코드가 많을수록 키 반복이 커집니다",
        "json-nested": "들여쓰기가 통째로 사라집니다",
        "kv-space": "콜론을 맞추려고 채운 공백이 전부 장식입니다",
        "md-table": "정렬 패딩과 구분선이 사라집니다",
        "log-repeat": "접두사는 길지만 줄 수가 적으면 이득도 적습니다",
        "xml": "태그 이름 자체는 못 줄입니다",
     }.get(k, "")]
     for k, v in sorted(m["by_kind"].items(), key=lambda x: -x[1]["saved"])],
    align=["left", "right", "right", "left"],
    title="유형별 절감 (structure 조건)",
)

print("적용 횟수:", m["applied_count"])

## 정리

- **무손실은 표현의 중복을 먹습니다** — 내용이 아니라 포맷을 줄입니다
- **입력이 결과를 정합니다** — 같은 코드가 28% 도 되고 1% 도 됩니다
- **검증할 수 없으면 무손실이 아닙니다** — 되돌리기나 정규형 비교 중 하나는 있어야 합니다
- **공백 접기는 대개 손해입니다** — 조금 더 얻고 보장을 잃습니다

무손실은 여기까지가 천장입니다. 더 줄이려면 **무언가는 버려야** 합니다.

### 다음 랩

[`02-handle-ref`](../02-handle-ref/run.ipynb) — 버리는 대신 밖에 두고
필요할 때만 꺼냅니다.